# QMML Hackathon - Support Session Demo
## Stock 1 Walkthrough

This notebook walks through the full pipeline for Stock 1:
1. Load & explore the data
2. Train/validation split
3. Fit a Linear Regression baseline
4. Fit XGBoost
5. Compare with RMSE
6. Make final prediction on the test set
7. Think about how prediction → spread

Then it's your turn to apply this to stocks 2–9!


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor


## 2. Load & Explore the Data

Let's start by understanding what we're working with.

In [2]:
# Load stock 1 training data
df = pd.read_csv('hackathon_data/stock_1_train.csv')
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Features: {df.shape[1] - 1}, Target: 1")
df.head()

Shape: (19999, 6)
Rows: 19999, Features: 5, Target: 1


,col_0,col_1,col_2,col_3,col_4,target
0,1.574674,0.170111,-0.045997,0.758763,0.185289,223.453581
1,-0.802095,-0.246373,1.640907,-0.141742,-0.163337,211.253687
2,0.216712,0.468679,-0.786830,-0.152416,0.293351,280.980886
3,0.864578,-0.343601,-1.563815,0.621198,1.124796,266.650511
4,1.465125,-0.861386,-0.950072,-0.218991,-0.597203,216.206945


### Quick sanity checks
- How are the features distributed?
- Any missing values?
- What range is the target in?

In [3]:
# Summary statistics
df.describe().round(2)

,col_0,col_1,col_2,col_3,col_4,target
count,19999.00,19999.00,19999.00,19999.00,19999.00,19999.00
mean,-0.01,-0.00,-0.00,0.00,-0.00,246.14
std,1.01,1.01,0.99,1.01,1.01,39.05
min,-3.89,-3.82,-4.39,-3.70,-4.26,80.00
25%,-0.68,-0.68,-0.67,-0.67,-0.68,219.84
50%,-0.02,-0.01,-0.00,-0.00,-0.01,246.11
75%,0.66,0.67,0.66,0.67,0.68,272.19
max,5.01,4.09,4.33,4.51,4.15,400.00


In [4]:
# Any missing values?
print(df.isnull().sum())

col_0     0
col_1     0
col_2     0
col_3     0
col_4     0
target    0
dtype: int64


In [5]:
# Which features correlate most with the target?
print(df.corr()['target'].sort_values(ascending=False))

target    1.000000
col_1     0.646885
col_4     0.000560
col_3    -0.002781
col_0    -0.385041
col_2    -0.642064
Name: target, dtype: float64


### Quick aside: the datasets are NOT all the same

In [6]:
# Check the shape of ALL 9 training sets
for i in range(1, 10):
    temp = pd.read_csv(f'hackathon_data/stock_{i}_train.csv')
    print(f"Stock {i}: {temp.shape[0]:>6} rows, {temp.shape[1]-1:>2} features")

Stock 1:  19999 rows,  5 features
Stock 2:   1499 rows, 15 features
Stock 3:     29 rows,  4 features
Stock 4:   9999 rows, 12 features
Stock 5:    799 rows, 20 features
Stock 6:    119 rows,  8 features
Stock 7:  19999 rows, 25 features
Stock 8:   1999 rows, 25 features
Stock 9:     59 rows, 20 features


Some stocks have 29 rows. Others have 20,000. A complex model will overfit badly on the small ones. Keep this in mind!

## 3. Train/Validation Split

This is important. The test set has only 1 row and no target, meaning we can't evaluate on it.

We must split the training data to create our own validation set. 
This is how we know how good our model is **before** the competition.

In [7]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# 80/20 split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Validation set: {X_val.shape[0]} rows")

Training set: 15999 rows
Validation set: 4000 rows


## 4. Linear Regression - The Baseline

Simple, fast, hard to overfit. Always start here.

In [8]:
# Fit Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on validation set
lr_preds = lr_model.predict(X_val)

# Calculate RMSE
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_preds))
print(f"Linear Regression RMSE: {lr_rmse:.2f}")

Linear Regression RMSE: 4.81


In [9]:
# What coefficients did LR learn?
for name, coef in zip(X.columns, lr_model.coef_):
    print(f"{name}: {coef:>8.2f}")
print(f"Intercept: {lr_model.intercept_:.2f}")

col_0:   -15.33
col_1:    25.22
col_2:   -25.37
col_3:     0.03
col_4:     0.10
Intercept: 245.95


## 5. XGBoost - Trying a different model

XGBoost can capture non-linear patterns and feature interactions that Linear Regression misses.

In [ ]:
# Fit XGBoost
xgb_model = XGBRegressor(
    n_estimators=100,   # number of trees
    max_depth=3,         # how complex each tree is
    learning_rate=0.1,   # how much each tree contributes
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Predict on validation set
xgb_preds = xgb_model.predict(X_val)

# Calculate RMSE
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_preds))
print(f"XGBoost RMSE: {xgb_rmse:.2f}")

## 6. Compare Models

In [ ]:
print("=" * 40)
print("Model Comparison — Stock 1")
print("=" * 40)
print(f"Linear Regression RMSE: {lr_rmse:.2f}")
print(f"XGBoost RMSE:           {xgb_rmse:.2f}")
print()

if lr_rmse < xgb_rmse:
    print(">>> Linear Regression wins on this stock!")
    print("This suggests the relationship is mostly linear.")
    print("A fancier model isn't always better.")
else:
    print(">>> XGBoost wins on this stock!")
    print("There are non-linear patterns that LR can't capture.")

**Key takeaway:** More complex ≠ always better. Test both, use what works. This will vary across the 9 stocks!

## 7. Make Final Prediction on the Test Set

Now we retrain on ALL the training data (no holdout) and predict the single test row.

In [ ]:
# Retrain on FULL training data
final_model = LinearRegression()  # using LR since it won on this stock
final_model.fit(X, y)

# Load the test set (just 1 row, no target)
test = pd.read_csv('hackathon_data/stock_1_test.csv')
print(f"Test set shape: {test.shape}")
print(test)

# Predict!
prediction = final_model.predict(test)[0]
print(f"\nPredicted true value for Stock 1: £{prediction:.2f}")


## 8. From Prediction to Spread

Your prediction + your RMSE = your spread.

RMSE tells you how uncertain your model is. Use it to set sensible bounds.

In [ ]:
print(f"Prediction: £{prediction:.2f}")
print(f"Model RMSE: £{lr_rmse:.2f}")
print()
print("Possible spread strategies:")
print(f"  Conservative (±2 RMSE): [{prediction - 2*lr_rmse:.2f}, {prediction + 2*lr_rmse:.2f}]  Spread = £{4*lr_rmse:.2f}")
print(f"  Moderate     (±1 RMSE): [{prediction - lr_rmse:.2f}, {prediction + lr_rmse:.2f}]  Spread = £{2*lr_rmse:.2f}")
print(f"  Aggressive (±0.5 RMSE): [{prediction - 0.5*lr_rmse:.2f}, {prediction + 0.5*lr_rmse:.2f}]  Spread = £{lr_rmse:.2f}")
print()
print("Tighter spread = more chance of becoming Market Maker = more risk!")


## 9. Reusable Functions

Here are some helper functions to speed up your workflow for the remaining stocks.

In [ ]:
def load_and_split(stock_num, test_size=0.2, random_state=42):
    """Load a stock's training data and split into train/val."""
    df = pd.read_csv(f'hackathon_data/stock_{stock_num}_train.csv')
    X = df.drop('target', axis=1)
    y = df['target']
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    return X, y, X_train, X_val, y_train, y_val


def evaluate(model, X_val, y_val):
    """Calculate RMSE for a fitted model."""
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))


def predict_test(model, stock_num):
    """Load the test set and return the single prediction."""
    test = pd.read_csv(f'hackathon_data/stock_{stock_num}_test.csv')
    return model.predict(test)[0]

## 10. Your Turn: Stocks 2 to 9

Apply what you've learned! Some things to think about:
- The datasets vary a lot in size (29 rows to 20,000 rows) and number of features (4 to 25)
- XGBoost might overfit on the smaller datasets — compare train vs val RMSE
- Try other models: Random Forest, Ridge/Lasso, LightGBM, neural nets
- Consider cross-validation instead of a single split, especially for small datasets
- Your RMSE directly informs your spread width — use it!

In [ ]:
# Stock 2


In [ ]:
# Stock 3


In [ ]:
# Stock 4


In [ ]:
# Stock 5


In [ ]:
# Stock 6


In [ ]:
# Stock 7


In [ ]:
# Stock 8


In [ ]:
# Stock 9
